# F1 Pit-Stop Prediction — Vanilla ML + Requested Feature Engineering

## Feature engineering performed

This version **keeps `Driver`** and adds exactly the seven requested features.

1. **TyreLife / LapNumber** → `TyreLife_LapNumber_Ratio`  
   Captures tyre age relative to the current lap.

2. **TyreLife × RaceProgress** → `TyreLife_RaceProgress`  
   Captures the combined effect of tyre age and race progression.

3. **LapNumber²** → `LapNumber_Squared`  
   Allows a nonlinear lap-number effect.

4. **RaceProgress²** → `RaceProgress_Squared`  
   Allows a nonlinear race-progress effect.

5. **Position × RaceProgress** → `Position_RaceProgress`  
   Captures the interaction between current position and race progress.

6. **TyreLife × Stint** → `TyreLife_Stint`  
   Captures how tyre age behaves within a stint.

7. **DistanceToFinish** → `1 - RaceProgress`  
   Represents the remaining fraction of the race.

### What is NOT done

- No `f1_strategy_dataset_v4.csv`
- No AutoGluon / AutoML
- No target encoding
- No lag/rolling features
- No additional interactions
- No additional polynomial features
- No scaling
- `Driver` is **not dropped**
- The original competition features are retained

The only new columns are the seven listed above.


## 1. Imports and configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

RANDOM_STATE = 42
VALID_SIZE = 0.20
TARGET = "PitNextLap"

# TRAIN_PATH = "train(3).csv"
# TEST_PATH = "test(3).csv"
# SUBMISSION_PATH = "sample_submission(3).csv"

TRAIN_PATH = r"D:\Work\DSA_Coding_Questions\F1_Pit_prediction\data\train.csv"
TEST_PATH = r"D:\Work\DSA_Coding_Questions\F1_Pit_prediction\data\test.csv"   
SUBMISSION_PATH = r"D:\Work\DSA_Coding_Questions\F1_Pit_prediction\data\sample_submission.csv"


## 2. Load only the supplied competition files

In [2]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SUBMISSION_PATH)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)


Train shape: (439140, 16)
Test shape: (188165, 15)
Sample submission shape: (188165, 2)


## 3. Basic cleanup — keep Driver

In [3]:
train = train.drop(columns=["id"])
test = test.drop(columns=["id"])

print("Driver retained:", "Driver" in train.columns)


Driver retained: True


## 4. Requested feature engineering

In [4]:
def add_requested_features(df):
    df = df.copy()

    df["TyreLife_LapNumber_Ratio"] = np.where(
        df["LapNumber"] != 0,
        df["TyreLife"] / df["LapNumber"],
        np.nan
    )

    df["TyreLife_RaceProgress"] = df["TyreLife"] * df["RaceProgress"]
    df["LapNumber_Squared"] = df["LapNumber"] ** 2
    df["RaceProgress_Squared"] = df["RaceProgress"] ** 2
    df["Position_RaceProgress"] = df["Position"] * df["RaceProgress"]
    df["TyreLife_Stint"] = df["TyreLife"] * df["Stint"]
    df["DistanceToFinish"] = 1 - df["RaceProgress"]

    return df

train = add_requested_features(train)
test = add_requested_features(test)

ENGINEERED_FEATURES = [
    "TyreLife_LapNumber_Ratio",
    "TyreLife_RaceProgress",
    "LapNumber_Squared",
    "RaceProgress_Squared",
    "Position_RaceProgress",
    "TyreLife_Stint",
    "DistanceToFinish",
]

print("Exactly these 7 engineered features were added:")
for f in ENGINEERED_FEATURES:
    print(" -", f)


Exactly these 7 engineered features were added:
 - TyreLife_LapNumber_Ratio
 - TyreLife_RaceProgress
 - LapNumber_Squared
 - RaceProgress_Squared
 - Position_RaceProgress
 - TyreLife_Stint
 - DistanceToFinish


## 5. Final feature set

In [5]:
BASE_FEATURES = [
    "Driver", "Compound", "Race", "Year", "PitStop", "LapNumber",
    "Stint", "TyreLife", "Position", "LapTime (s)", "LapTime_Delta",
    "Cumulative_Degradation", "RaceProgress", "Position_Change"
]

FEATURES = BASE_FEATURES + ENGINEERED_FEATURES

X = train[FEATURES].copy()
y = train[TARGET].copy()
X_test = test[FEATURES].copy()

print("Original features:", len(BASE_FEATURES))
print("Engineered features:", len(ENGINEERED_FEATURES))
print("Total features:", len(FEATURES))
print("\nFinal features:")
for i, f in enumerate(FEATURES, 1):
    print(f"{i:2d}. {f}")


Original features: 14
Engineered features: 7
Total features: 21

Final features:
 1. Driver
 2. Compound
 3. Race
 4. Year
 5. PitStop
 6. LapNumber
 7. Stint
 8. TyreLife
 9. Position
10. LapTime (s)
11. LapTime_Delta
12. Cumulative_Degradation
13. RaceProgress
14. Position_Change
15. TyreLife_LapNumber_Ratio
16. TyreLife_RaceProgress
17. LapNumber_Squared
18. RaceProgress_Squared
19. Position_RaceProgress
20. TyreLife_Stint
21. DistanceToFinish


## 6. Feature-engineering sanity check

In [6]:
display(train[ENGINEERED_FEATURES].describe().T)
print("\nMissing values in engineered features:")
display(train[ENGINEERED_FEATURES].isnull().sum().to_frame("missing"))


,count,mean,std,min,25%,50%,75%,max
TyreLife_LapNumber_Ratio,439140.0,0.788312,0.428675,0.014493,0.489796,1.000000,1.000000,25.000000
TyreLife_RaceProgress,439140.0,6.326401,7.655756,0.012821,0.901408,3.372549,9.038462,76.012821
LapNumber_Squared,439140.0,821.465020,1011.830445,1.000000,81.000000,361.000000,1296.000000,6084.000000
RaceProgress_Squared,439140.0,0.178164,0.225230,0.000164,0.016866,0.072485,0.263331,1.000000
Position_RaceProgress,439140.0,3.261060,3.292479,0.012821,0.771930,2.083333,4.736111,20.000000
TyreLife_Stint,439140.0,26.799668,25.089277,1.000000,8.000000,18.000000,39.000000,365.000000
DistanceToFinish,439140.0,0.662339,0.253277,0.000000,0.486842,0.730769,0.870130,0.987179



Missing values in engineered features:


,missing
TyreLife_LapNumber_Ratio,0
TyreLife_RaceProgress,0
LapNumber_Squared,0
RaceProgress_Squared,0
Position_RaceProgress,0
TyreLife_Stint,0
DistanceToFinish,0


## 7. Train/validation split

In [7]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=VALID_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

print(X_train.shape, X_valid.shape)


(351312, 21) (87828, 21)


## 8. Categorical preprocessing

`Driver`, `Compound`, and `Race` are categorical.

LightGBM/CatBoost receive categorical variables directly. XGBoost/Random Forest/Extra Trees use ordinal encoding with safe handling for unseen categories. This is preprocessing, not additional feature engineering.


In [8]:
CATEGORICAL_FEATURES = ["Driver", "Compound", "Race"]
NUMERICAL_FEATURES = [c for c in FEATURES if c not in CATEGORICAL_FEATURES]

preprocessor = ColumnTransformer([
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
    ]), CATEGORICAL_FEATURES),
    ("num", SimpleImputer(strategy="median"), NUMERICAL_FEATURES)
])

X_train_encoded = preprocessor.fit_transform(X_train)
X_valid_encoded = preprocessor.transform(X_valid)
X_test_encoded = preprocessor.transform(X_test)


## 9. LightGBM

In [9]:
lgbm = LGBMClassifier(
    objective="binary", n_estimators=5000, learning_rate=0.03,
    num_leaves=31, max_depth=-1, subsample=0.8,
    colsample_bytree=0.8, reg_alpha=0.0, reg_lambda=1.0,
    random_state=RANDOM_STATE, n_jobs=-1
)

Xtr = X_train.copy()
Xva = X_valid.copy()

for c in CATEGORICAL_FEATURES:
    combined = pd.concat([Xtr[c], Xva[c]]).astype("category")
    cats = combined.cat.categories
    Xtr[c] = pd.Categorical(Xtr[c], categories=cats)
    Xva[c] = pd.Categorical(Xva[c], categories=cats)

lgbm.fit(
    Xtr, y_train,
    categorical_feature=CATEGORICAL_FEATURES,
    eval_set=[(Xva, y_valid)],
    callbacks=[early_stopping(150, verbose=False), log_evaluation(0)]
)

pred_lgbm = lgbm.predict_proba(Xva)[:, 1]
auc_lgbm = roc_auc_score(y_valid, pred_lgbm)
print(f"LightGBM ROC-AUC: {auc_lgbm:.6f}")


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 69905, number of negative: 281407
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.039462 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3315
[LightGBM] [Info] Number of data points in the train set: 351312, number of used features: 21
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.198983 -> initscore=-1.392665
[LightGBM] [Info] Start training from score -1.392665
LightGBM ROC-AUC: 0.944643


## 10. XGBoost

In [10]:
xgb = XGBClassifier(
    objective="binary:logistic", eval_metric="auc",
    n_estimators=5000, learning_rate=0.03, max_depth=7,
    min_child_weight=1, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.0, reg_lambda=1.0, tree_method="hist",
    random_state=RANDOM_STATE, n_jobs=-1
)

xgb.fit(X_train_encoded, y_train, eval_set=[(X_valid_encoded, y_valid)], verbose=False)
pred_xgb = xgb.predict_proba(X_valid_encoded)[:, 1]
auc_xgb = roc_auc_score(y_valid, pred_xgb)
print(f"XGBoost ROC-AUC: {auc_xgb:.6f}")


XGBoost ROC-AUC: 0.948421


## 11. CatBoost

In [11]:
Xtr_cb = X_train.copy()
Xva_cb = X_valid.copy()

for c in CATEGORICAL_FEATURES:
    Xtr_cb[c] = Xtr_cb[c].fillna("__MISSING__").astype(str)
    Xva_cb[c] = Xva_cb[c].fillna("__MISSING__").astype(str)

catboost = CatBoostClassifier(
    loss_function="Logloss", eval_metric="AUC",
    iterations=5000, learning_rate=0.03, depth=8,
    l2_leaf_reg=3.0, random_seed=RANDOM_STATE,
    verbose=False, thread_count=-1, allow_writing_files=False
)

catboost.fit(
    Xtr_cb, y_train, cat_features=CATEGORICAL_FEATURES,
    eval_set=(Xva_cb, y_valid), use_best_model=True,
    early_stopping_rounds=150, verbose=False
)

pred_cat = catboost.predict_proba(Xva_cb)[:, 1]
auc_cat = roc_auc_score(y_valid, pred_cat)
print(f"CatBoost ROC-AUC: {auc_cat:.6f}")


CatBoost ROC-AUC: 0.950566


## 12. Random Forest

In [12]:
rf = RandomForestClassifier(
    n_estimators=700, criterion="entropy", max_depth=None,
    min_samples_split=2, min_samples_leaf=1, max_features="sqrt",
    bootstrap=True, random_state=RANDOM_STATE, n_jobs=-1
)
rf.fit(X_train_encoded, y_train)
pred_rf = rf.predict_proba(X_valid_encoded)[:, 1]
auc_rf = roc_auc_score(y_valid, pred_rf)
print(f"Random Forest ROC-AUC: {auc_rf:.6f}")


Random Forest ROC-AUC: 0.944408


## 13. Extra Trees

In [13]:
extra = ExtraTreesClassifier(
    n_estimators=300,
    criterion="entropy",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    bootstrap=False,
    random_state=RANDOM_STATE,
    n_jobs=4
)
extra.fit(X_train_encoded, y_train)

pred_extra = extra.predict_proba(X_valid_encoded)[:, 1]

auc_extra = roc_auc_score(y_valid, pred_extra)

print(f"Extra Trees ROC-AUC: {auc_extra:.6f}")


Extra Trees ROC-AUC: 0.938561


## 14. Compare models

In [14]:
results = pd.DataFrame({
    "Model": ["LightGBM", "XGBoost", "CatBoost", "RandomForest", "ExtraTrees"],
    "Validation_ROC_AUC": [auc_lgbm, auc_xgb, auc_cat, auc_rf, auc_extra]
}).sort_values("Validation_ROC_AUC", ascending=False)

display(results)


,Model,Validation_ROC_AUC
2,CatBoost,0.950566
1,XGBoost,0.948421
0,LightGBM,0.944643
3,RandomForest,0.944408
4,ExtraTrees,0.938561


## 15. Fixed weighted vanilla ensemble

In [15]:
weights = {
    "LightGBM": 0.30,
    "XGBoost": 0.25,
    "CatBoost": 0.20,
    "RandomForest": 0.15,
    "ExtraTrees": 0.10
}

pred_ensemble = (
    weights["LightGBM"] * pred_lgbm +
    weights["XGBoost"] * pred_xgb +
    weights["CatBoost"] * pred_cat +
    weights["RandomForest"] * pred_rf +
    weights["ExtraTrees"] * pred_extra
)

auc_ensemble = roc_auc_score(y_valid, pred_ensemble)

print("Weights:", weights)
print(f"Ensemble ROC-AUC: {auc_ensemble:.6f}")


Weights: {'LightGBM': 0.3, 'XGBoost': 0.25, 'CatBoost': 0.2, 'RandomForest': 0.15, 'ExtraTrees': 0.1}
Ensemble ROC-AUC: 0.950269


## 16. Retrain final models on all training data

In [16]:
X_full = train[FEATURES].copy()
y_full = train[TARGET].copy()
X_test_full = test[FEATURES].copy()

# LightGBM
Xfull_lgb = X_full.copy()
Xtest_lgb = X_test_full.copy()

for c in CATEGORICAL_FEATURES:
    combined = pd.concat([Xfull_lgb[c], Xtest_lgb[c]]).astype("category")
    cats = combined.cat.categories
    Xfull_lgb[c] = pd.Categorical(Xfull_lgb[c], categories=cats)
    Xtest_lgb[c] = pd.Categorical(Xtest_lgb[c], categories=cats)

lgbm_final = LGBMClassifier(
    objective="binary",
    n_estimators=int(lgbm.best_iteration_ or 1000),
    learning_rate=0.03, num_leaves=31, max_depth=-1,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.0, reg_lambda=1.0,
    random_state=RANDOM_STATE, n_jobs=-1
)
lgbm_final.fit(Xfull_lgb, y_full, categorical_feature=CATEGORICAL_FEATURES,
               callbacks=[log_evaluation(0)])
final_lgbm = lgbm_final.predict_proba(Xtest_lgb)[:, 1]

# Shared numeric/categorical preprocessing
final_preprocessor = ColumnTransformer([
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
    ]), CATEGORICAL_FEATURES),
    ("num", SimpleImputer(strategy="median"), NUMERICAL_FEATURES)
])
Xfull_enc = final_preprocessor.fit_transform(X_full)
Xtest_enc = final_preprocessor.transform(X_test_full)

# XGBoost
xgb_final = XGBClassifier(
    objective="binary:logistic", eval_metric="auc",
    n_estimators=xgb.get_booster().num_boosted_rounds(),
    learning_rate=0.03, max_depth=7, min_child_weight=1,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.0, reg_lambda=1.0, tree_method="hist",
    random_state=RANDOM_STATE, n_jobs=-1
)
xgb_final.fit(Xfull_enc, y_full, verbose=False)
final_xgb = xgb_final.predict_proba(Xtest_enc)[:, 1]

# CatBoost
Xfull_cb = X_full.copy()
Xtest_cb = X_test_full.copy()
for c in CATEGORICAL_FEATURES:
    Xfull_cb[c] = Xfull_cb[c].fillna("__MISSING__").astype(str)
    Xtest_cb[c] = Xtest_cb[c].fillna("__MISSING__").astype(str)

cat_final = CatBoostClassifier(
    loss_function="Logloss", eval_metric="AUC",
    iterations=catboost.get_best_iteration() + 1,
    learning_rate=0.03, depth=8, l2_leaf_reg=3.0,
    random_seed=RANDOM_STATE, verbose=False,
    thread_count=-1, allow_writing_files=False
)
cat_final.fit(Xfull_cb, y_full, cat_features=CATEGORICAL_FEATURES, verbose=False)
final_cat = cat_final.predict_proba(Xtest_cb)[:, 1]

# Random Forest
rf_final = RandomForestClassifier(
    n_estimators=700, criterion="entropy", max_depth=None,
    min_samples_split=2, min_samples_leaf=1, max_features="sqrt",
    bootstrap=True, random_state=RANDOM_STATE, n_jobs=-1
)
rf_final.fit(Xfull_enc, y_full)
final_rf = rf_final.predict_proba(Xtest_enc)[:, 1]

# Extra Trees
extra_final = ExtraTreesClassifier(
    n_estimators=700, criterion="entropy", max_depth=None,
    min_samples_split=2, min_samples_leaf=1, max_features="sqrt",
    bootstrap=False, random_state=RANDOM_STATE, n_jobs=-1
)
extra_final.fit(Xfull_enc, y_full)
final_extra = extra_final.predict_proba(Xtest_enc)[:, 1]

print("Final models trained on all competition training rows.")

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 87381, number of negative: 351759
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.061456 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3314
[LightGBM] [Info] Number of data points in the train set: 439140, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.198982 -> initscore=-1.392668
[LightGBM] [Info] Start training from score -1.392668
Final models trained on all competition training rows.


## 17. Create submission

In [17]:
final_prediction = (
    weights["LightGBM"] * final_lgbm +
    weights["XGBoost"] * final_xgb +
    weights["CatBoost"] * final_cat +
    weights["RandomForest"] * final_rf +
    weights["ExtraTrees"] * final_extra
)

submission = sample_submission.copy()
submission[TARGET] = final_prediction

OUTPUT_PATH = "best_vanilla_feature_engineered.csv"
submission.to_csv(OUTPUT_PATH, index=False)

assert len(submission) == len(test)
assert submission[TARGET].between(0, 1).all()

print("Saved:", OUTPUT_PATH)
print("Submission shape:", submission.shape)
display(submission.head())

Saved: best_vanilla_feature_engineered.csv
Submission shape: (188165, 2)


,id,PitNextLap
0,439140,0.006274
1,439141,0.005733
2,439142,0.003828
3,439143,0.189817
4,439144,0.878645
